In [194]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.linear_model import SGDRegressor
from sklearn.preprocessing import StandardScaler

In [2]:
X,y = load_diabetes(return_X_y=True)

In [65]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2 ,random_state=42)

In [195]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [277]:
class MeraRidgeSGDSimplified:
    def __init__(self,epoch,learning_rate,alpha):
        self.coef_ = None
        self.intercept_ = None
        self.epoch = epoch
        self.learning_rate = learning_rate
        self.alpha = alpha
    
    def fit(self,X_train,y_train):
        self.coef_ = np.ones(X_train.shape[1])
        self.intercept_ = 0
        weights = np.insert(self.coef_,0,self.intercept_)
        X_train = np.insert(X_train,0,1,axis=1)

        for i in range(self.epoch):
            indices = np.random.permutation(X_train.shape[0])
            for idx in indices: 
                penalty = self.alpha * weights.copy()
                penalty[0] = 0
                error = np.dot(X_train[idx],weights) - y_train[idx]
                der = np.dot(X_train[idx].T,error) + penalty
                weights = weights - self.learning_rate * der

        self.intercept_ = weights[0]
        self.coef_ = weights[1:]
    
    def predict(self,X_test):
        return np.dot(X_test,self.coef_) + self.intercept_


__Second Version of the Formulae__

In [278]:
class MeraRidgeSGD:
    def __init__(self,epoch,learning_rate,alpha):
        self.coef_ = None
        self.intercept_ = None
        self.epoch = epoch
        self.learning_rate = learning_rate
        self.alpha = alpha
    
    def fit(self,X_train,y_train):
        self.coef_ = np.ones(X_train.shape[1])
        self.intercept_ = 0
        weights = np.insert(self.coef_,0,self.intercept_)
        X_train = np.insert(X_train,0,1,axis=1)

        for i in range(self.epoch):
            for _ in range(X_train.shape[0]):
                idx = np.random.randint(X_train.shape[0])
                penalty = self.alpha * weights.copy()
                penalty[0] = 0
                der = np.outer(X_train[idx], X_train[idx]).dot(weights) - X_train[idx] * y_train[idx] + penalty
                weights = weights - self.learning_rate * der

        self.intercept_ = weights[0]
        self.coef_ = weights[1:]
    
    def predict(self,X_test):
        return np.dot(X_test,self.coef_) + self.intercept_

In [271]:
ridge = MeraRidgeSGD(epoch = 10000 , learning_rate= 0.01 , alpha = 0.5)
ridge.fit(X_train_scaled,y_train)
y_pred_cp = ridge.predict(X_test_scaled)

In [279]:
from sklearn.metrics import r2_score
from sklearn.linear_model import SGDRegressor

# My Simplified SGD
ridge_s = MeraRidgeSGDSimplified(epoch=1000, learning_rate=0.01, alpha=0.5)
ridge_s.fit(X_train_scaled, y_train)
y_pred_yours = ridge_s.predict(X_test_scaled)

# My SGD Variation
ridge = MeraRidgeSGD(epoch = 1000 , learning_rate= 0.01 , alpha = 0.5)
ridge.fit(X_train_scaled,y_train)
y_pred_cp = ridge.predict(X_test_scaled)

# Sklearn SGDRegressor (fair comparison)
sklearn_sgd = SGDRegressor(
    penalty='l2',
    alpha=0.5,
    learning_rate='constant',
    eta0=0.01,
    max_iter=10000,
    shuffle=True,
    fit_intercept=True
)
sklearn_sgd.fit(X_train_scaled, y_train)
y_pred_sklearn = sklearn_sgd.predict(X_test_scaled)


# R2 comparison
print("My SGD Simplified     → R2:", r2_score(y_test, y_pred_yours))
print("My SGD Complex        → R2:",r2_score(y_test,y_pred_cp))
print("Sklearn SGD  → R2:", r2_score(y_test, y_pred_sklearn))

# Coef comparison
print("\nMy coef Simplified  :", ridge_s.coef_)
print("\nMy coef Comlpex     :", ridge.coef_)
print("Sklearn coef:", sklearn_sgd.coef_)

print("\nMy intercept Simplified  :", ridge_s.intercept_)
print("\nMy intercept Complex     :", ridge.intercept_)
print("Sklearn intercept:", sklearn_sgd.intercept_)

My SGD Simplified     → R2: 0.4106328578855961
My SGD Complex        → R2: 0.4200239721061355
Sklearn SGD  → R2: 0.41195247240103094

My coef Simplified  : [ 1.92782964 -5.2423047  23.13680956 14.91942756 -4.33307958 -7.32949752
 -5.99400508  1.66104567 13.1970181   6.73429412]

My coef Comlpex     : [ 7.60747514 -9.00386068 23.76812703 11.78705692  1.0986624   3.01049483
 -8.65431746  5.53392679 10.81828447  1.81389112]
Sklearn coef: [ 3.88188438 -3.01486813 15.99735363 12.82921293 -1.64042282 -2.65012225
 -6.67729595  5.44729454 11.22526307  1.72144823]

My intercept Simplified  : 150.37906014333672

My intercept Complex     : 153.10506301328323
Sklearn intercept: [150.30424435]
